# 01 Generate Synthetic Datasets

This notebook creates small local supervised fine-tuning datasets for the finance, legal, and healthcare adapters.

```mermaid
flowchart LR
    A[Domain templates] --> B[Synthetic chat records]
    B --> C[datasets/generated/finance.jsonl]
    B --> D[datasets/generated/legal.jsonl]
    B --> E[datasets/generated/healthcare.jsonl]
```

The records use chat-style `system`, `user`, and `assistant` messages so the same files can be consumed by the Qwen chat template during LoRA training.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PROJECT_SPEC.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

from llmops_demo.settings import settings, ensure_dirs

cfg = settings()
print(f"Project root: {PROJECT_ROOT}")
print(f"Base model: {cfg.base_model}")
print(f"Adapters: {cfg.adapters}")

## Generate datasets

Expected output: three JSONL files under `datasets/generated/`, one per adapter.

In [ ]:
import importlib.util

module_path = PROJECT_ROOT / "datasets" / "generate_synthetic.py"
spec = importlib.util.spec_from_file_location("generate_synthetic", module_path)
generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator)

records_per_domain = int(os.getenv("SYNTHETIC_RECORDS_PER_DOMAIN", "60"))
ensure_dirs(cfg.dataset_dir)

for adapter in cfg.adapters:
    rows = generator.generate_domain(adapter, records_per_domain)
    output_path = cfg.dataset_dir / f"{adapter}.jsonl"
    generator.write_jsonl(output_path, rows)
    print(f"{adapter}: wrote {len(rows)} records to {output_path}")

## Inspect a sample

The sample should show the domain-specific system prompt and a concise assistant answer.

In [ ]:
import json

for adapter in cfg.adapters:
    path = cfg.dataset_dir / f"{adapter}.jsonl"
    first = json.loads(path.read_text(encoding="utf-8").splitlines()[0])
    print(f"\n[{adapter}] {first['id']}")
    for message in first["messages"]:
        print(f"{message['role']}: {message['content'][:160]}")